In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Extract 33 landmarks

In [ ]:
!pip install -q "protobuf>=5.29.1,<6.0.0" mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 7.2 MB/s eta 0:00:00


In [3]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import glob
import numpy as np
import urllib.request

print("="*60)
print("MEDIAPIPE POSE ESTIMATION - 33 LANDMARKS")
print("="*60)

PIE_PATH = "/content/drive/MyDrive/PIE"

# Download the pose landmarker model (only runs once)
MODEL_PATH = "/content/pose_landmarker_full.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading pose landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
        MODEL_PATH
    )
    print("Model downloaded.")

# Initialize with new Tasks API
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
detector = vision.PoseLandmarker.create_from_options(options)

# Find all extracted frame folders
frame_folders = glob.glob(f"{PIE_PATH}/images_annotated/**/video_*", recursive=True)
print(f"Found {len(frame_folders)} video folders to process")

total_frames_processed = 0
total_landmarks_saved = 0

for folder_idx, folder in enumerate(frame_folders):
    print(f"\n[{folder_idx+1}/{len(frame_folders)}] Processing: {os.path.basename(folder)}")

    frames = sorted(glob.glob(f"{folder}/*.jpg"))
    print(f"   Frames: {len(frames)}")

    output_folder = folder.replace("images_annotated", "landmarks")
    os.makedirs(output_folder, exist_ok=True)

    frames_with_landmarks = 0

    for frame_path in frames:
        # New API uses mp.Image directly
        try:
            mp_image = mp.Image.create_from_file(frame_path)
        except Exception:
            total_frames_processed += 1
            continue

        results = detector.detect(mp_image)

        if results.pose_landmarks:
            # results.pose_landmarks is a list of poses; take the first
            landmarks = []
            for lm in results.pose_landmarks[0]:
                landmarks.extend([lm.x, lm.y, lm.z])
            # 99 values (33 × 3), same as before

            frame_name = os.path.basename(frame_path).replace('.jpg', '.npy')
            save_path = os.path.join(output_folder, frame_name)
            np.save(save_path, np.array(landmarks))

            frames_with_landmarks += 1
            total_landmarks_saved += 1

        total_frames_processed += 1

        if total_frames_processed % 1000 == 0:
            print(f"   Processed {total_frames_processed} frames total...")

    print(f"   Landmarks found in {frames_with_landmarks}/{len(frames)} frames")
    print(f"   Saved to: {output_folder}")

print(f"\n" + "="*60)
print(f"COMPLETE!")
print(f"   Total frames processed: {total_frames_processed}")
print(f"   Total landmark files saved: {total_landmarks_saved}")
print("="*60)

MEDIAPIPE POSE ESTIMATION - 33 LANDMARKS
Model downloaded.
Found 53 video folders to process

[1/53] Processing: video_0004
   Frames: 2166
   Processed 1000 frames total...
   Processed 2000 frames total...
   Landmarks found in 133/2166 frames
   Saved to: /content/drive/MyDrive/PIE/landmarks/set01/video_0004

[2/53] Processing: video_0001
   Frames: 2020
   Processed 3000 frames total...
   Processed 4000 frames total...
   Landmarks found in 86/2020 frames
   Saved to: /content/drive/MyDrive/PIE/landmarks/set01/video_0001

[3/53] Processing: video_0002
   Frames: 6862
   Processed 5000 frames total...
   Processed 6000 frames total...
   Processed 7000 frames total...
   Processed 8000 frames total...
   Processed 9000 frames total...
   Processed 10000 frames total...
   Processed 11000 frames total...
   Landmarks found in 1031/6862 frames
   Saved to: /content/drive/MyDrive/PIE/landmarks/set01/video_0002

[4/53] Processing: video_0003
   Frames: 8063
   Processed 12000 frames to

In [ ]:
import numpy as np

data = np.load('/content/drive/MyDrive/PIE/landmarks/set01/video_0001/frame_01088.npy')

print(data)
print(type(data))
print(data.shape)


[ 0.32272145  0.69894779  0.03462816  0.32771111  0.69724071  0.02849472
  0.32814392  0.69786453  0.02847269  0.32856396  0.69863391  0.0284593
  0.32477432  0.6935358   0.04131217  0.32460448  0.69342864  0.04130316
  0.32439449  0.69347811  0.04122003  0.33381999  0.70103717  0.01260282
  0.32794547  0.69936496  0.07167203  0.32441726  0.70849079  0.02951211
  0.32370532  0.70769292  0.04659225  0.34611091  0.75646794 -0.02633367
  0.31996521  0.7420373   0.09697673  0.31306279  0.79917014 -0.04432095
  0.28412217  0.77143008  0.09872493  0.2826544   0.7953583  -0.03955299
  0.24813977  0.76204348  0.06240762  0.27463779  0.79746324 -0.04616525
  0.23802313  0.75863028  0.06113786  0.2757718   0.79049778 -0.04671175
  0.23737535  0.75493968  0.04728045  0.27799478  0.7901758  -0.03931507
  0.24189274  0.75857759  0.05472621  0.30401766  0.86302298 -0.04753929
  0.29140204  0.84435046  0.04755051  0.26882741  0.80798239 -0.11563182
  0.26640582  0.76374573  0.01200517  0.20600088  0.